In [1]:
# colocate mode velocity reward training debug notebook

# 1. load model & dataset
# 2. initialize PerTokenAdvantageTrainer (vllm colocate mode )
# 3. train with log. 
# Issues: (1) no reward logging for correct rollout 

In [1]:
# ── 1. Setup ────────────────────────────────────────────────────────────────
# Run from repo root. Keeps everything in a scratch dir so we can rm -rf safely.
import os, sys, json, shutil, random
from pathlib import Path

REPO = Path("/Users/fangyuanyu/Implementation/arl")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

OUT = REPO / "output" / "debug_vrl"
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)
print("scratch dir:", OUT)

# Toggle: vLLM colocate is what the bug was observed under, but the velocity-
# log code path is identical with use_vllm=False (HF generate). Flip to True
# on a CUDA box to reproduce in the exact training config.
USE_VLLM   = False
MODEL_NAME = "Qwen/Qwen3-0.6B"
N_STEPS    = 3
N_GEN      = 4              # rollouts per prompt
PDB        = 2              # per_device_train_batch_size
GAS        = 2              # gradient_accumulation_steps
MAX_COMP   = 256
SEED       = 0
random.seed(SEED)

import numpy as np, torch
np.random.seed(SEED); torch.manual_seed(SEED)
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())


scratch dir: /Users/fangyuanyu/Implementation/arl/output/debug_vrl
torch: 2.9.1 cuda: False


In [2]:
# ── 2. Dataset (tiny) + model/tokenizer ─────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM
from src.game24utils import (
    build_puzzle_pool, bucket_by_difficulty, make_splits, build_datasets,
    correctness_reward, format_reward,
)

puzzles = build_puzzle_pool(max_n=9)
easy, medium, hard = bucket_by_difficulty(puzzles, easy_min=8, hard_max=2)
train_p, eval_p, hard_probe = make_splits(easy, medium, hard, eval_frac=0.10, probe_frac=0.40)
train_ds, eval_ds, _ = build_datasets(train_p, eval_p, hard_probe)

# Shrink to keep the debug fast — we only need a few prompts.
train_ds = train_ds.select(range(min(16, len(train_ds))))
print("train_ds rows:", len(train_ds))
print("sample row:", train_ds[0])

tok   = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)
print("loaded:", MODEL_NAME)


train_ds rows: 16
sample row: {'prompt': [{'content': "You play the Game of 24. Given four numbers, you must write a single arithmetic expression using each number exactly once with + - * / and parentheses that evaluates to 24.\n\nThink step by step. First reason about how to combine the numbers, then on the final line output only the expression after '#### '.\nExample final line: '#### (3+5)*(7-4)'.", 'role': 'system'}, {'content': "Given numbers: 3,3,5,9. Make 24. Think step by step, then give your final expression on the last line after '#### '.", 'role': 'user'}], 'numbers': [3, 3, 5, 9], 'solutions': ['((3+5)/3)*9', '(3+5)/(3/9)', '((3+5)*9)/3', '(3+5)*(9/3)', '3*(5+(9/3))', '(9+3)*(5-3)', '(9*(3+5))/3', '9*((3+5)/3)', '((9/3)+5)*3', '(9/3)*(5+3)', '9/(3/(5+3))', '3*((9/3)+5)', '((5+3)*9)/3', '(5+3)*(9/3)', '(5-3)*(9+3)', '(5+(9/3))*3', '(3+3)*(9-5)', '(9/3)*(3+5)', '9/(3/(3+5))', '(9-5)*(3+3)', '(9*(5+3))/3', '9*((5+3)/3)', '((5+3)/3)*9', '(5+3)/(3/9)', '(5-3)*(3+9)', '(3+9)*(5-3

`torch_dtype` is deprecated! Use `dtype` instead!


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

loaded: Qwen/Qwen3-0.6B


In [3]:
# ── 3. Seed the answer buffer with dataset-provided solutions ───────────────
# Mirrors ablate_game24.py:319-325. Without seeding, buffer.sample returns
# empty for every query → the else-branch ref is empty → ALL rollouts get
# silently skipped at the gate. That would explain an empty velocity_log
# but NOT the observed pattern (0 correct out of 3200 logged), so we want a
# populated buffer to isolate the *correct-rollout* path.
from src.online_buffer import OnlineBuffer
from src.velocity import VelocityRewardComputer

answer_buffer = OnlineBuffer(capacity_per_query=64)
n_seed = 0
for row in train_ds:
    qk = tuple(row["numbers"])
    for expr in (row.get("solutions") or []):
        n_seed += int(answer_buffer.add(qk, expr))
print(f"buffer seeded: {n_seed} entries / {answer_buffer.num_queries()} queries")

vel_computer = VelocityRewardComputer(
    answer_buffer,
    chunk_strategy="uniform",
    chunk_size=16,
    normalize_by_chunk=True,
)


buffer seeded: 412 entries / 16 queries


In [4]:
# ── 3. Seed the answer buffer with dataset-provided solutions ───────────────
# Mirrors ablate_game24.py:319-325. Without seeding, buffer.sample returns
# empty for every query → the else-branch ref is empty → ALL rollouts get
# silently skipped at the gate. That would explain an empty velocity_log
# but NOT the observed pattern (0 correct out of 3200 logged), so we want a
# populated buffer to isolate the *correct-rollout* path.
from src.online_buffer import OnlineBuffer
from src.velocity import VelocityRewardComputer

answer_buffer = OnlineBuffer(capacity_per_query=64)
n_seed = 0
for row in train_ds:
    qk = tuple(row["numbers"])
    for expr in (row.get("solutions") or []):
        n_seed += int(answer_buffer.add(qk, expr))
print(f"buffer seeded: {n_seed} entries / {answer_buffer.num_queries()} queries")

vel_computer = VelocityRewardComputer(
    answer_buffer,
    chunk_strategy="uniform",
    chunk_size=16,
    normalize_by_chunk=True,
)


buffer seeded: 412 entries / 16 queries


In [5]:
# ── 4. Instrumented is_correct + extract_answer ──────────────────────────────
# Wrap the two functions whose silent failures we want to attribute. Every
# call is appended to a sidecar JSONL so we can replay them after training and
# see exactly which rollouts the trainer thought were correct, and what
# extract_answer returned on them.
from script.ablate_game24 import game24_is_correct, game24_query_key
from src.velocity import _default_extract_answer

PROBE_LOG = OUT / "probe.jsonl"
PROBE_LOG.write_text("")

def _append_probe(rec):
    with PROBE_LOG.open("a") as f:
        f.write(json.dumps(rec) + "\n")

def is_correct_probed(completion_str: str, prompt_str: str) -> bool:
    ok    = game24_is_correct(completion_str, prompt_str)
    ext   = _default_extract_answer(completion_str)
    nums  = []
    try:
        import re
        m = re.search(r"Given numbers:\s*([\d,\s]+)", prompt_str)
        if m:
            nums = [int(x) for x in m.group(1).split(",") if x.strip()]
    except Exception:
        pass
    _append_probe({
        "is_correct":     bool(ok),
        "extract_answer": ext,
        "nums_parsed":    nums,
        "prompt_str_head": prompt_str[:200],
        "prompt_str_tail": prompt_str[-200:],
        "completion_tail": completion_str[-200:],
    })
    return ok


objc[71234]: Class AVFFrameReceiver is implemented in both /Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/av/.dylibs/libavdevice.61.1.100.dylib (0x2f1d14798) and /Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x306fe83a8). One of the two will be used. Which one is undefined.
objc[71234]: Class AVFAudioReceiver is implemented in both /Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/av/.dylibs/libavdevice.61.1.100.dylib (0x2f1d147e8) and /Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x306fe83f8). One of the two will be used. Which one is undefined.


In [6]:
# ── 5. Build the trainer ────────────────────────────────────────────────────
from trl import GRPOConfig
from src.pertoken_trainer import PerTokenAdvantageTrainer
from script.ablate_game24 import RolloutLogger

rollout_logger = RolloutLogger(OUT / "rollouts.jsonl", OUT / "eval_rollouts.jsonl", tok)

cfg_kw = dict(
    output_dir=str(OUT),
    num_generations=N_GEN,
    max_completion_length=MAX_COMP,
    per_device_train_batch_size=PDB,
    gradient_accumulation_steps=GAS,
    learning_rate=5e-6,
    max_steps=N_STEPS,
    logging_steps=1,
    bf16=True,
    save_strategy="no",
    report_to="none",
    use_vllm=USE_VLLM,
)
if USE_VLLM:
    cfg_kw.update(vllm_mode="colocate", vllm_gpu_memory_utilization=0.4,
                  vllm_max_model_length=max(MAX_COMP + 512, 1024))
cfg = GRPOConfig(**cfg_kw)

trainer = PerTokenAdvantageTrainer(
    model=model,
    reward_funcs=[correctness_reward, format_reward, rollout_logger],
    args=cfg,
    train_dataset=train_ds,
    processing_class=tok,
    adv_mode="token", adv_n_chunks=8, adv_stride=5,
    velocity_computer=vel_computer,
    is_correct=is_correct_probed,         # ← instrumented
    query_key_fn=game24_query_key,
)
trainer.velocity_log_path = OUT / "velocity_log.jsonl"
trainer.velocity_log_path.write_text("")
print("trainer ready; velocity_log →", trainer.velocity_log_path)


[2026-05-26 11:44:24,619] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to mps (auto detect)


W0526 11:44:24.755000 71234 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


trainer ready; velocity_log → /Users/fangyuanyu/Implementation/arl/output/debug_vrl/velocity_log.jsonl


In [ ]:
# ── 6. Train a few steps ────────────────────────────────────────────────────
trainer.train()
print("done. files in", OUT, ":")
for p in sorted(OUT.iterdir()):
    print(f"  {p.name:30s}  {p.stat().st_size:>8d} bytes")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
# ── 7. Cross-check rollouts.jsonl ↔ velocity_log.jsonl correctness counts ───
def load_jsonl(p):
    return [json.loads(l) for l in Path(p).read_text().splitlines() if l.strip()]

ro   = load_jsonl(OUT / "rollouts.jsonl")
vel  = load_jsonl(OUT / "velocity_log.jsonl")
prob = load_jsonl(PROBE_LOG)

print(f"rollouts.jsonl     : {len(ro):4d}  correct={sum(r['correct'] for r in ro)}")
print(f"velocity_log.jsonl : {len(vel):4d}  correct={sum(r['correct'] for r in vel)}")
print(f"probe.jsonl (is_correct calls): {len(prob):4d}  "
      f"is_correct=True={sum(p['is_correct'] for p in prob)}  "
      f"extract_answer!=None={sum(p['extract_answer'] is not None for p in prob)}  "
      f"nums_parsed_ok={sum(bool(p['nums_parsed']) for p in prob)}")


In [ ]:
# ── 8. Localize the failure: which gate kills the correct rollouts? ─────────
# Tally each probe call against the four possible disagreement patterns.
# This pinpoints which line in velocity.py:603-608 is doing the silent skip.
from collections import Counter

states = Counter()
for p in prob:
    states[(bool(p["nums_parsed"]), p["is_correct"], p["extract_answer"] is not None)] += 1

print("(nums_parsed_ok, is_correct, extract_answer_ok)  →  count")
for k, v in sorted(states.items(), key=lambda kv: -kv[1]):
    print(f"  {k}  →  {v}")

# Show one prompt where nums regex failed, if any.
bad = [p for p in prob if not p["nums_parsed"]]
if bad:
    print("\nFirst prompt where _nums_from_prompt regex failed:")
    print("  head:", repr(bad[0]["prompt_str_head"]))
    print("  tail:", repr(bad[0]["prompt_str_tail"]))
else:
    print("\nAll probe prompts had nums parsed OK.")


In [ ]:
# ── 9. extract_answer post-mortem on actual completions ─────────────────────
# For every rollout that rollouts.jsonl marks correct, ask: does the velocity
# computer's default regex find an answer in that exact completion?
import re
from src.velocity import _DEFAULT_ANSWER_RE

correct_rollouts = [r for r in ro if r["correct"]]
print(f"rollouts.jsonl correct: {len(correct_rollouts)}")

n_match = 0
sample_miss = None
for r in correct_rollouts:
    # rollouts.jsonl stores skip_special_tokens=True text via tok.encode/decode
    # convention; what the velocity computer actually sees is decoded with
    # skip_special_tokens=False from completion_ids. Re-encode/decode to mirror.
    ids = tok.encode(r["completion"], add_special_tokens=False)
    comp_str = tok.decode(ids, skip_special_tokens=False)
    m = _DEFAULT_ANSWER_RE.search(comp_str.strip())
    if m:
        n_match += 1
    elif sample_miss is None:
        sample_miss = comp_str

print(f"extract_answer would match on {n_match}/{len(correct_rollouts)} of those")
if sample_miss is not None:
    print("\nFirst miss (re-encoded completion tail):")
    print(repr(sample_miss[-300:]))


## What to read off

After running cells 1-9, the diagnosis is determined by the table in cell 8:

| `(nums_parsed_ok, is_correct, extract_answer_ok)` | meaning |
|---|---|
| `(False, False, *)` rows dominate | **`_nums_from_prompt` regex bug** — `is_correct` returns `False` for everything, all rollouts go to the buffer-sample else branch, log shows 0 correct. Fix: pass dataset `numbers` through instead of re-parsing the prompt. |
| `(True, True, False)` rows non-zero | **`extract_answer` silently drops correct rollouts** at `velocity.py:607-608`. Fix: fall back to buffer.sample when extract_answer returns None. |
| `(True, True, True)` rows match `velocity_log` correct count | Pipeline is OK — bug is elsewhere. |

Cell 9 independently verifies whether the default regex `_DEFAULT_ANSWER_RE` can find an answer in the correct rollouts' completion text. If `n_match < len(correct_rollouts)`, that's the smoking gun for hypothesis 2.

The two findings are **not exclusive**: the regex bug masks hypothesis 2 because the `if correctness[b]:` branch is never taken when `is_correct` always returns `False`. Fixing `is_correct` first will likely *expose* hypothesis 2.
